In [1]:
import json
from pathlib import Path
import os

In [4]:
main_dir = Path().resolve().parent

In [6]:
if os.getcwd() == main_dir:
    pass
else:
    os.chdir(main_dir)

In [ ]:
try:
    from src.utils import sql_parser  # when run from project root
except ImportError:
    from utils import sql_parser      # when run directly as src/BFS.py

# 1. BFS code

In [9]:
CATS_DICT_PATH = main_dir / "data/input/final_categories.json"
TARGET_CATS_PATH = main_dir / "data/output/target_categories.json"

In [49]:
def load_cat_list(categories_dict_path: Path) -> dict:
    with open(categories_dict_path, "r") as f:
        cats_dict = json.load(f)
    return cats_dict

In [50]:
categories_list = load_cat_list(CATS_DICT_PATH)

In [51]:
categories_list

[{'_comment': 'This is a note for developers',
  'Info': 'Structure follows https://en.wikipedia.org/wiki/Wikipedia:Contents/Categories. Duplicates resolved by keeping categories in their most general/primary sector.',
  'source': 'https://en.wikipedia.org/wiki/Wikipedia:Contents/Categories',
  'enabled': True},
 {'field_name': 'Mathematics and logic',
  'techs_related': 1,
  'field_id': 1,
  'subfields': [{'subfield_name': 'Mathematics',
    'techs_related': 1,
    'subfield_id': 11,
    'subsubfields': ['Algebra',
     'Geometry',
     'Calculus',
     'Arithmetic',
     'Trigonometry',
     'Mathematical analysis',
     'Number theory',
     'Combinatorics',
     'Topology',
     'Mathematical logic']},
   {'subfield_name': 'Applied mathematics',
    'techs_related': 1,
    'subfield_id': 12,
    'subsubfields': ['Computational science',
     'Operations research',
     'Numerical analysis',
     'Mathematical optimization',
     'Dynamical systems',
     'Chaos theory']},
   {'subf

In [57]:
subsubfield_to_look = "Political philosophy"
tech_related_subfields = []

for field_dict in categories_list:
    # print(field_dict)
    if 'subfields' in field_dict:
        subfields_list = field_dict['subfields'] 
        for subfield_dict in subfields_list:
            if subfield_dict['techs_related'] == 1:
                if 'subsubfields' in subfield_dict:
                    tech_related_subfields += subfield_dict['subsubfields']
                
    # for subfield in field:
        # print(subfield['subfield_name'])
#     #     subfield = {k for k,v in field.items() if v == "Political philosophy"}
#     # print(subfield)

In [59]:
# tech_related_subfields

In [60]:
def tech_related(subsubfield: str, tech_related_subfields_list: list)->bool:
    """
    Inser a subsubfield and return if it's tech related according to final_categories categories_dict
    some entire fields are technical
    some 
    """
    return (subsubfield in tech_related_subfields)

In [62]:
tech_related('Topology', tech_related_subfields)

True

In [10]:
# algorithm to find the target categories
def BFS(page_id: int, target_cats) -> tuple[set, int]:
    counter = 1
    intersection = dict()
    old_categories = set()
    visited = set()
    UNIQUE_TOP = False

    while counter <= 5 and not UNIQUE_TOP:
        if counter == 1:
            new_categories = sql_parser.get_categories_for_page(page_id)
        else:
            subcats_ids = sql_parser.get_subcats_ids_for_cats(old_categories)
            new_categories = set()
            for x in subcats_ids:
                new_categories.update(sql_parser.get_categories_for_page(x))

        new_intersection = sql_parser.get_intersection_cat_target(new_categories, target_cats)
        if new_intersection:
            weight = 1 / counter  # depth 1 = 1.0, depth 2 = 0.5, depth 3 = 0.33 ...
            for inter in new_intersection:
                intersection[inter] = intersection.get(inter, 0) + weight

        old_categories = new_categories - visited
        if not old_categories:
            break
        visited.update(new_categories)

        if intersection:
            max_count = max(intersection.values())
            top = {k for k, v in intersection.items() if v == max_count}
            if counter == 1 and len(top) == 1:           # 1st iteration: unique winner (the counter was increased before, so it's effectively the 1st)
                UNIQUE_TOP = True
            elif 2 <= counter <= 4 and len(top) <= 2:          # 2nd iteration and others: at most 2
                UNIQUE_TOP = True
            elif counter >= 5 and len(top) <= 2 and max_count > 0.5:  # 5th+ iteration: at most 2 with evidence
                UNIQUE_TOP = True

        counter += 1

    depth = counter - 1

    if not intersection:
        return {"Other"}, depth
    return top, depth

In [ ]:
page_id_smartphone = 167079
page_id = page_id_smartphone

In [19]:
BFS(page_id_smartphone,target_cats)

{'Computer engineering', 'Computer science', 'Consumer electronics'}

In [20]:
BFS_hard(page_id_smartphone,target_cats)

Checking layer 5
current dictionary {'Consumer electronics': 3, 'Embedded systems': 2, 'Computer science': 2, 'Manufacturing': 2, 'Classes of computers': 1, 'Computer engineering': 2, 'Application software': 2, 'Mechanical engineering': 1, 'Electrical engineering': 1, 'Electronics manufacturing': 1, 'Information systems': 1, 'Computer architecture': 1, 'Software': 1, 'Telecommunications': 1}
Checking layer 10
current dictionary {'Consumer electronics': 3, 'Embedded systems': 2, 'Computer science': 4, 'Manufacturing': 3, 'Classes of computers': 3, 'Computer engineering': 4, 'Application software': 2, 'Mechanical engineering': 3, 'Electrical engineering': 5, 'Electronics manufacturing': 1, 'Information systems': 3, 'Computer architecture': 3, 'Software': 2, 'Telecommunications': 5, 'Management': 1, 'Control theory': 3, 'Automation': 1, 'Human–computer interaction': 3, 'Machines': 2, 'Systems engineering': 2, 'Internet': 2, 'Information technology': 3, 'Software engineering': 3, 'Digital 

{'Computer science', 'Electrical engineering', 'Telecommunications'}

In [461]:
BFS(page_id_nn,target_cats)

{'Biological engineering', 'Biotechnology', 'Medicine', 'Software engineering'}

In [462]:
BFS_hard(page_id_nn,target_cats)

Checking layer 5
current dictionary {'Biological engineering': 2, 'Biotechnology': 1, 'Medical technology': 1, 'Medicine': 1, 'Artificial intelligence': 1, 'Software engineering': 1}
Total layers checked: 6
{'Biological engineering': 2, 'Biotechnology': 2, 'Medical technology': 1, 'Medicine': 2, 'Artificial intelligence': 1, 'Software engineering': 3, 'Information technology': 2, 'Data': 2, 'Control theory': 1, 'Computer engineering': 1, 'Computer science': 1, 'Systems engineering': 2, 'Electrical engineering': 1, 'Management': 1, 'Information systems': 1, 'Manufacturing': 1}


{'Software engineering'}

In [463]:
page_id_gpu = 390214
BFS(page_id_gpu,target_cats)

{'Computer architecture',
 'Computer engineering',
 'Digital electronics',
 'Digital media',
 'Human–computer interaction',
 'Integrated circuits'}

In [464]:
BFS_hard(page_id_gpu,target_cats)

Checking layer 5
current dictionary {'Electronic design': 2, 'Artificial intelligence': 1, 'Digital electronics': 3, 'Integrated circuits': 3, 'Computer engineering': 3, 'Human–computer interaction': 3, 'Electrical engineering': 1, 'Digital media': 2, 'Computer architecture': 2, 'Consumer electronics': 1, 'Information systems': 1, 'Computer science': 1, 'Electrical components': 1, 'Information technology': 1, 'Application software': 1, 'Multimedia': 1, 'Machines': 1, 'Tools': 1, 'Manufacturing': 1, 'Systems engineering': 1, 'Semiconductors': 1}
Total layers checked: 9
{'Electronic design': 2, 'Artificial intelligence': 1, 'Digital electronics': 3, 'Integrated circuits': 3, 'Computer engineering': 4, 'Human–computer interaction': 3, 'Electrical engineering': 5, 'Digital media': 4, 'Computer architecture': 4, 'Consumer electronics': 1, 'Information systems': 3, 'Computer science': 3, 'Electrical components': 4, 'Information technology': 4, 'Application software': 2, 'Multimedia': 1, 'Mac

{'Electrical engineering'}

In [465]:
# transistors 30011
page_id_transistor = 30011

In [466]:
BFS(page_id_transistor,target_cats)

{'Electrical components', 'Electrical engineering'}

In [467]:
BFS_hard(page_id_transistor,target_cats)

Checking layer 5
current dictionary {'Electrical components': 3, 'Electrical engineering': 3, 'Semiconductors': 1, 'Manufacturing': 2, 'Materials science': 1, 'Energy': 1, 'Computer engineering': 1, 'Mechanical engineering': 1, 'Computer science': 1, 'Systems engineering': 1}
Total layers checked: 8
{'Electrical components': 5, 'Electrical engineering': 6, 'Semiconductors': 1, 'Manufacturing': 4, 'Materials science': 1, 'Energy': 1, 'Computer engineering': 4, 'Mechanical engineering': 4, 'Computer science': 3, 'Systems engineering': 1, 'Telecommunications': 4, 'Machines': 2, 'Tools': 1, 'Civil engineering': 2, 'Construction': 2, 'Electronics manufacturing': 1, 'Companies': 3, 'Architecture': 2, 'Information technology': 1, 'Computer architecture': 1, 'Classes of computers': 1, 'Data': 1, 'Management': 1}


{'Electrical engineering'}

In [468]:
page_id_ml = 233488

In [469]:
BFS(page_id_ml,target_cats)

{'Information technology'}

In [470]:
BFS_hard(page_id_ml,target_cats)

Checking layer 5
current dictionary {'Artificial intelligence': 1, 'Software engineering': 1, 'Control theory': 1, 'Data': 1, 'Information technology': 1, 'Systems engineering': 1, 'Computer engineering': 1}
Total layers checked: 5
{'Artificial intelligence': 1, 'Software engineering': 1, 'Control theory': 1, 'Data': 1, 'Information technology': 2, 'Systems engineering': 1, 'Computer engineering': 1, 'Electrical engineering': 1, 'Computer science': 1}


{'Information technology'}

In [471]:
page_dna_fingerprints = 44290

In [472]:
BFS(page_dna_fingerprints,target_cats)

{'Biotechnology',
 'Forensic science',
 'Human–computer interaction',
 'Information technology',
 'Medicine',
 'Tools'}

In [473]:
BFS_hard(page_dna_fingerprints,target_cats)

Checking layer 5
current dictionary {'Forensic science': 2, 'Data': 1, 'Human–computer interaction': 1, 'Information technology': 1, 'Medicine': 1, 'Biotechnology': 1, 'Tools': 1, 'Biological engineering': 1, 'Systems engineering': 1}
Total layers checked: 9
{'Forensic science': 2, 'Data': 4, 'Human–computer interaction': 3, 'Information technology': 5, 'Medicine': 4, 'Biotechnology': 3, 'Tools': 4, 'Biological engineering': 2, 'Systems engineering': 3, 'Software engineering': 4, 'Artificial intelligence': 2, 'Computer science': 4, 'Machines': 1, 'Computer architecture': 3, 'Information systems': 2, 'Application software': 1, 'Materials science': 2, 'Telecommunications': 3, 'Mechanical engineering': 2, 'Computer security': 1, 'Computer engineering': 4, 'Manufacturing': 2, 'Military science': 2, 'Electrical engineering': 3, 'Digital media': 2, 'Architecture': 3, 'Software': 1, 'Management': 2, 'Multimedia': 1, 'Construction': 1, 'Energy': 1, 'Operating systems': 1, 'Internet': 1, 'Medic

{'Information technology'}

In [26]:
get_page_title(1145960)

'Vehicle_registration_plates_of_Italy'

In [28]:
get_page_title(1145960)

'Vehicle_registration_plates_of_Italy'

In [25]:
print(find_lt_title_id(30011))

None
